# Magnetized Metalic Objects - One-Dimensional Galerkin

## Import Packages 

In [1]:
using StaticArrays   # provides statically typed arrays 
using SparseArrays   # provides sparse arrays 

## Section 1: Introduction 

Provides 1D Galerkin FEM code wioth linear elements. More later. 

<b>Exercises</b>
1. validate the code showing quadratic reduction of error in max-norm given analytical solution;
2. extend to 2D / 3D;
3. extend with an integral term;

## Section 2: Entire Code Base 

In [17]:
"""
    struct Elem1DLin  

Holds information for a single linear line finite element. 
"""
struct Elem1DLin
  inode::SVector{2,Int64} # element global node tag 
  len::Float64            # element length  
end; 

"""
    struct Mesh 

Holds information for the entire mesh as an array of linear line elements. 
"""
struct Mesh 
  nodes::Array{Float64,1}      # 1D array of nodes 
  elements::Array{Elem1DLin,1} # 1D array of Elem1DLin  
end; 

function genVec(mesh, sourcefct)
    
    Np1 = length(mesh.elements)+1
    nodes = mesh.nodes 
    
    f = zeros(Np1)

    for (k,element) in enumerate(mesh.elements) 
        f[element.inode] += element.len*sourcefct.(nodes[element.inode])
    end 
   
    return f 
    
end;

function genMatDense(mesh)
    
    Np1 = length(mesh.elements)+1
    nodes = mesh.nodes 
    
    A = zeros(Np1,Np1)

    for (k,element) in enumerate(mesh.elements) 
        inode = element.inode
        A[inode,inode] += 1/element.len*[1. -1.;-1. 1.]
    end 
   
    return A 
    
end; 

function genMat(mesh)
    
    nelems      = length(mesh.elements)
    dofPerElem  = length(mesh.elements[1].inode)
    dofPerElem2 = dofPerElem^2 
    
    Avals = zeros(Float64,dofPerElem2*nelems)
    I = zeros(Int64,length(Avals))
    J = zeros(Int64,length(Avals))

    irng = SVector{dofPerElem2}(1:dofPerElem2)
    
    for (k,element) in enumerate(mesh.elements) 
        Ielem = SVector{dofPerElem2}(element.inode[i] for j=1:dofPerElem, i=1:dofPerElem) 
        Jelem = SVector{dofPerElem2}(element.inode[i] for i=1:dofPerElem, j=1:dofPerElem)
        Aelem = 1/(element.len)*SVector{dofPerElem2}(1.,-1., -1., 1.) 
        I[irng] .= Ielem  
        J[irng] .= Jelem  
        Avals[irng] .= Aelem    
        irng = irng.+dofPerElem2    
    end 
   
    A = sparse(I,J,Avals)
    
    return A 
    
end; 

## Section 3: Sample Usage 

### Mesh Generation and Elementary Verifications  

In [18]:
# set uniform mesh with N cells and h = 1/N meshwidth  
N = 4; h = 1/N; Np1 = N+1; 

# set the pionts in the mesh 
nodes = Vector(0:h:1); 

# set the mesh 
mesh = Mesh(nodes, [Elem1DLin([k,k+1], nodes[k+1]-nodes[k]) for k=1:N]); 

In [19]:
sum([elem.len for elem in mesh.elements])

sum(map(x -> x.len, mesh.elements))
    
mapreduce(x -> x.len, +, mesh.elements)

1.0

In [20]:
# extract nodes and elements from mesh 
(; nodes, elements) = mesh 
display(nodes)
display(elements)

5-element Vector{Float64}:
 0.0
 0.25
 0.5
 0.75
 1.0

4-element Vector{Elem1DLin}:
 Elem1DLin([1, 2], 0.25)
 Elem1DLin([2, 3], 0.25)
 Elem1DLin([3, 4], 0.25)
 Elem1DLin([4, 5], 0.25)

### Generate Load Vector 

In [21]:
sourcefct(x) = x^2 
sourcefct.(nodes[mesh.elements[2].inode])

2-element SVector{2, Float64} with indices SOneTo(2):
 0.0625
 0.25

In [22]:
f = zeros(Np1)

for (k,element) in enumerate(mesh.elements) 
    f[element.inode] += element.len*sourcefct.(nodes[element.inode])
end 

display(f)

5-element Vector{Float64}:
 0.0
 0.03125
 0.125
 0.28125
 0.25

In [23]:
f = genVec(mesh, sourcefct)

5-element Vector{Float64}:
 0.0
 0.03125
 0.125
 0.28125
 0.25

In [24]:
#@code_warntype genVec(mesh, x->x^2)

### Generate Stiffness Matrix 

In [25]:
A = zeros(Np1,Np1)

for (k,element) in enumerate(mesh.elements) 
    A[element.inode,element.inode] += (1.0/element.len)*[1. -1.;-1. 1.]
end 

display(A)

5×5 Matrix{Float64}:
  4.0  -4.0   0.0   0.0   0.0
 -4.0   8.0  -4.0   0.0   0.0
  0.0  -4.0   8.0  -4.0   0.0
  0.0   0.0  -4.0   8.0  -4.0
  0.0   0.0   0.0  -4.0   4.0

In [26]:
A = genMatDense(mesh)

5×5 Matrix{Float64}:
  4.0  -4.0   0.0   0.0   0.0
 -4.0   8.0  -4.0   0.0   0.0
  0.0  -4.0   8.0  -4.0   0.0
  0.0   0.0  -4.0   8.0  -4.0
  0.0   0.0   0.0  -4.0   4.0

In [27]:
A = genMat(mesh)

5×5 SparseMatrixCSC{Float64, Int64} with 13 stored entries:
  4.0  -4.0    ⋅     ⋅     ⋅ 
 -4.0   8.0  -4.0    ⋅     ⋅ 
   ⋅   -4.0   8.0  -4.0    ⋅ 
   ⋅     ⋅   -4.0   8.0  -4.0
   ⋅     ⋅     ⋅   -4.0   4.0